# Point of this Notebook

This runs the champion on the latest day for all 50 stocks and writes a daily_signals table
that the SQL views, Genie, Lakebase, and the app all read from. Normal compute while developing.

In [0]:
import mlflow
import pandas as pd
from pyspark.sql.functions import col, current_timestamp, lit, desc

CATALOG = "raghavan_trading_signals"
MODEL_NAME = f"{CATALOG}.ml.signal_model"
mlflow.set_registry_uri("databricks-uc")

champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
print("Loaded champion:", MODEL_NAME)

In [0]:
gold = spark.table(f"{CATALOG}.gold.daily_features")
latest_date = gold.agg({"trade_date": "max"}).collect()[0][0]
print("Scoring for:", latest_date)

metadata_cols = ["symbol","trade_date","open","high","low","close","adj_close","volume",
                 "dividends","stock_splits","prev_close","bronze_ingested_at",
                 "bronze_source_file","next_day_return","next_day_direction"]
feature_cols = [c for c in gold.columns if c not in metadata_cols]

## As it is an AutoML / tabular champion — score the latest day directly

In [0]:
latest = gold.filter(col("trade_date") == latest_date).toPandas()
preds = champion.predict(latest[feature_cols].fillna(0))

signals = pd.DataFrame({
    "symbol": latest["symbol"], "trade_date": latest["trade_date"],
    "close_price": latest["close"], "predicted_direction": preds,
    "rsi_14": latest["rsi_14"], "macd_histogram": latest["macd_histogram"],
    "bb_position": latest["bb_position"], "volatility_20d": latest["volatility_20d"],
    "volume_ratio": latest["volume_ratio"],
})
signals["signal"] = signals["predicted_direction"].map({1: "BUY", 0: "SELL"})
print(signals["signal"].value_counts())

## Write the signals table (both paths)

In [0]:
from delta.tables import DeltaTable

out = (spark.createDataFrame(signals)
       .withColumn("scored_at", current_timestamp())
       .withColumn("model_name", lit(MODEL_NAME)))

target = f"{CATALOG}.gold.daily_signals"

if not spark.catalog.tableExists(target):
    # first ever run — create the table
    out.write.format("delta").saveAsTable(target)
else:
    # idempotent upsert — replaces this day's rows, never appends duplicates
    (DeltaTable.forName(spark, target).alias("t")
        .merge(out.alias("s"), "t.symbol = s.symbol AND t.trade_date = s.trade_date")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

print("Upserted", out.count(), "signals into", target)

display(spark.table(target)
        .filter(col("trade_date") == latest_date)
        .orderBy(desc("volume_ratio")))

## Understand the DAG to be built

```
extract_market_data (01)
        → ingest_to_bronze (02)
              → transform_to_silver (DLT pipeline from Sprint 2)
                     ├── compute_gold_features (04) ──→ score_daily_signals (11)
                     ├── compute_gold_regimes (05)   (Vector index auto-syncs on trigger)
                     └── archive_to_iceberg (10)
```

Creating a workflow called `raghavan-trading-signals-daily`

> The AI Search index needs no task — it's a `TRIGGERED` Delta-Sync index. Either add a tiny
> task that calls `index.sync()` via the `AISearchClient`, or trigger it inside `05`. Optional
> for a daily cadence.